# Xuất điểm tư thế của 4 bài tập ra CSV

Tạo một file ZIP với cấu trúc `neck/Dung/`, `neck/Sai/`, `shoulder/Dung/`, `shoulder/Sai/`, `wrist_up/Dung/`, `wrist_up/Sai/`, `wrist_down/Dung/`, `wrist_down/Sai/`. Có thể chỉ đưa một vài bài vào ZIP. Ảnh JPG, PNG hoặc WebP; mỗi ảnh chứa một người thấy rõ khớp cần theo dõi. Nhãn **Dung/Sai do bạn tự phân loại**; MediaPipe chỉ lấy điểm, không xác định đúng sai. Chạy lần lượt các ô và chọn ZIP khi Colab hỏi. CSV chứa 33 điểm x/y/z/visibility (132 cột), cùng nhãn và tên bài. Notebook không huấn luyện mô hình và ứng dụng camera hiện chưa đọc CSV này.


In [ ]:
!pip -q install "mediapipe==0.10.21" pillow
import csv, io, pathlib, urllib.request, zipfile
import numpy as np
import mediapipe as mp
from PIL import Image, ImageOps
from google.colab import files

EXERCISES = {"neck", "shoulder", "biceps", "wrist_up", "wrist_down"}
LABELS = {"Dung", "Sai"}
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"
MODEL = pathlib.Path("/content/pose_landmarker_full.task")
if not MODEL.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL)
print("MediaPipe", mp.__version__)


In [ ]:
print("Chọn đúng 1 file ZIP chứa các thư mục bài tập / Dung hoặc Sai")
uploaded = files.upload()
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
    raise ValueError("Hãy tải lên đúng một file ZIP")
zip_name, zip_data = next(iter(uploaded.items()))
images = []
with zipfile.ZipFile(io.BytesIO(zip_data)) as archive:
    for entry in archive.infolist():
        if entry.is_dir():
            continue
        path = pathlib.PurePosixPath(entry.filename)
        if ".." in path.parts or path.is_absolute():
            raise ValueError("ZIP chứa đường dẫn không hợp lệ")
        matches = [(i, path.parts[i], path.parts[i+1]) for i in range(len(path.parts)-1)
                   if path.parts[i] in EXERCISES and path.parts[i+1] in LABELS]
        if not matches or path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        _, exercise, label = matches[-1]
        images.append((exercise, label, entry.filename, archive.read(entry)))
if not images:
    raise ValueError("Không thấy ảnh trong thư mục <bài tập>/<Dung hoặc Sai>/")
print("Ảnh tìm thấy:", len(images))


In [ ]:
columns = ["exercise", "label", "source_image"] + [f"p{i}_{axis}" for i in range(33) for axis in ("x", "y", "z", "visibility")]
rows, skipped = [], []
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE, num_poses=1)
with mp.tasks.vision.PoseLandmarker.create_from_options(options) as detector:
    for exercise, label, name, data in images:
        try:
            image = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")
            result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(image)))
            if not result.pose_landmarks or len(result.pose_landmarks[0]) != 33:
                skipped.append((name, "Không thấy đủ 33 điểm"))
                continue
            points = result.pose_landmarks[0]
            if min(points[i].visibility or 0 for i in (11, 12)) < 0.55:
                skipped.append((name, "Hai vai không rõ"))
                continue
            values = [value for p in points for value in (p.x, p.y, p.z, p.visibility or 0.0)]
            rows.append([exercise, label, name] + values)
        except Exception as error:
            skipped.append((name, type(error).__name__))
if not rows:
    raise ValueError("Không ảnh nào trích xuất được điểm; hãy chụp rõ toàn thân/phần thân cần tập")
out = pathlib.Path("/content/bodypoint_pose_landmarks.csv")
with out.open("w", newline="", encoding="utf-8-sig") as stream:
    writer = csv.writer(stream)
    writer.writerow(columns)
    writer.writerows(rows)
print("Đã xuất", len(rows), "ảnh; bỏ qua", len(skipped), "ảnh")
for item in skipped[:20]:
    print("Bỏ qua:", *item)
print("Số mẫu theo bài/nhãn:")
from collections import Counter
print(Counter((r[0], r[1]) for r in rows))
files.download(str(out))
